# 🛌 Sleep Quality Prediction — Clean, Leak-Free Pipeline
### Based on: *"Predicting sleep quality with digital biomarkers and ANN"* (Lee et al., 2025)

---
## ⚠️ What was wrong in the previous version (and what we fixed)

| Problem | Fix Applied |
|---|---|
| Random 80/20 split let same person appear in train AND test | **Subject-wise split** — no participant overlaps |
| ISI/PHQ9/GAD7/WHOQOL may reflect WASO indirectly (questionnaire leakage) | **Leakage audit cell** — tests with/without these features |
| Scaling done on full dataset before split | **Scaler fit only on train set**, applied to test |
| ARIMA per-participant used test-participant data | Fixed to held-out participants only |
| No shuffle-label sanity check | **Shuffle test** included |

## 📌 Neural Networks Used
Yes — three of the seven models are neural networks:
- **GRU** — Gated Recurrent Unit (RNN family)
- **TCN** — Temporal Convolutional Network
- **Transformer** — Multi-head self-attention
- **LSTM** — Long Short-Term Memory (best model from paper, also a neural network)

Non-neural baselines: ARIMA, Random Forest, XGBoost

**Runtime:** Enable T4 GPU → Runtime → Change runtime type → T4 GPU

## 📦 Cell 1 — Install dependencies

In [ ]:
!pip install -q numpy pandas scikit-learn xgboost matplotlib seaborn
!pip install -q tensorflow lime shap statsmodels scipy
print('✅ All packages installed.')

## 🔧 Cell 2 — Imports & Seeds

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              roc_auc_score, log_loss, confusion_matrix,
                              classification_report, roc_curve)
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBClassifier
from scipy import stats
from scipy.stats import kstest
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import lime, lime.lime_tabular
import shap

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
                     'axes.grid':True,'grid.alpha':0.3,'font.size':11})

print('✅ TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 🗂️ Cell 3 — Synthetic Dataset
> Mirrors paper Table 2 stats. ISI/PHQ9/GAD7/WHOQOL are **not** directly derived from WASO — they are sampled from independent distributions with only weak, realistic correlations, mimicking real-world psychology data.

In [ ]:
np.random.seed(SEED)

N_PARTICIPANTS   = 82
DAYS_PER_SUBJECT = 27
TOTAL_DAYS       = N_PARTICIPANTS * DAYS_PER_SUBJECT

pid_arr      = np.repeat(np.arange(N_PARTICIPANTS), DAYS_PER_SUBJECT)
day_arr      = np.tile(np.arange(DAYS_PER_SUBJECT), N_PARTICIPANTS)
day_of_week  = day_arr % 7
season_arr   = (pid_arr >= 41).astype(int)

# ── Per-participant stable baselines (individual differences) ─────────────
rng = np.random.default_rng(SEED)
p_lf_base    = rng.normal(0.60, 0.10, N_PARTICIPANTS)[pid_arr]
p_rmssd_base = rng.normal(135.0, 18.0, N_PARTICIPANTS)[pid_arr]

# ── Biometric features (wearable sensor data) ─────────────────────────────
lf_hf    = np.clip(p_lf_base  + rng.normal(0, 0.07, TOTAL_DAYS), 0.15, 1.5)
rmssd    = np.clip(p_rmssd_base + rng.normal(0, 18, TOTAL_DAYS),  30,  280)
steps    = np.clip(rng.lognormal(np.log(4200), 0.85, TOTAL_DAYS), 0, 35000)
accel    = np.clip(rng.normal(1.5, 4.5, TOTAL_DAYS), 0, 25)
gyro     = np.clip(rng.normal(0.5, 9.0, TOTAL_DAYS), 0, 45)
lum      = np.clip(rng.lognormal(np.log(500), 0.85, TOTAL_DAYS), 0, 5000)

# ── Questionnaire features ─────────────────────────────────────────────────
# Sampled independently from biometric data.
# They have a *weak* real-world correlation with WASO but are NOT derived from it.
isi_base    = rng.gamma(1.9, 3.1, N_PARTICIPANTS)       # mean ~5.9
whoqol_base = rng.normal(99.4, 13.0, N_PARTICIPANTS)     # mean ~99.4
phq9_base   = rng.gamma(0.5, 3.4, N_PARTICIPANTS)        # mean ~1.7
gad7_base   = rng.gamma(0.4, 2.8, N_PARTICIPANTS)        # mean ~1.1

isi_d, whoqol_d, phq9_d, gad7_d, knhanes_d = [],[],[],[],[]
for p in range(N_PARTICIPANTS):
    for d in range(DAYS_PER_SUBJECT):
        bw = min(d // 14, 2)
        w  = min(d // 7,  4)
        isi_d.append(np.clip(isi_base[p] + rng.normal(0, 0.5), 0, 28))
        whoqol_d.append(np.clip(whoqol_base[p] + rng.normal(0, 2), 26, 130))
        phq9_d.append(np.clip(phq9_base[p] + rng.normal(0, 0.3), 0, 27))
        gad7_d.append(np.clip(gad7_base[p] + rng.normal(0, 0.3), 0, 21))
        knhanes_d.append(np.clip(rng.normal(28.7, 10.3), 0, 60))

isi_arr     = np.array(isi_d)
whoqol_arr  = np.array(whoqol_d)
phq9_arr    = np.array(phq9_d)
gad7_arr    = np.array(gad7_d)
knhanes_arr = np.array(knhanes_d)

# ── WASO target ────────────────────────────────────────────────────────────
# Built from sensor features + weak questionnaire influence + real noise.
# Noise std=5.5 makes this a genuinely hard problem (realistic).
waso = (
    6.5
    + 7.5  * lf_hf              # main HRV predictor (paper: r=0.22)
    - 0.015 * rmssd             # higher RMSSD → less stress → less WASO
    + 0.25  * isi_arr           # ISI has WEAK real influence on WASO
    - 0.025 * whoqol_arr        # better QoL → slightly less WASO
    - 0.0002 * steps            # more activity → slightly better sleep
    + 0.04  * accel
    + rng.normal(0, 5.5, TOTAL_DAYS)   # substantial irreducible noise
)
# Weekend effect
waso[day_of_week >= 5] += rng.uniform(0.5, 2.5, (day_of_week >= 5).sum())
waso = np.clip(waso, 0, 38)

# Binary label — threshold at 41.5th percentile (paper: class0=41.5%, class1=58.5%)
thr          = np.percentile(waso, 41.5)
waso_binary  = (waso > thr).astype(int)

df = pd.DataFrame({
    'pid': pid_arr, 'day': day_arr, 'dow': day_of_week, 'season': season_arr,
    'lf_hf': lf_hf, 'rmssd': rmssd, 'steps': steps,
    'accel': accel, 'gyro': gyro, 'lum': lum,
    'isi': isi_arr, 'whoqol': whoqol_arr, 'phq9': phq9_arr,
    'gad7': gad7_arr, 'knhanes': knhanes_arr,
    'waso_min': waso, 'waso': waso_binary
})

# Introduce 3% missing in sensor cols (realistic)
for col in ['lf_hf','rmssd','steps','accel','gyro','lum']:
    miss = rng.choice(len(df), int(0.03*len(df)), replace=False)
    df.loc[miss, col] = np.nan

print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} cols')
vc = df['waso'].value_counts(normalize=True)
print(f'Class balance: 0={vc[0]*100:.1f}%  1={vc[1]*100:.1f}%  (paper: 41.5% / 58.5%)')
print(f'WASO: mean={df.waso_min.mean():.1f}±{df.waso_min.std():.1f}  (paper: 13.1±6.5)')

# Verify questionnaire correlations are weak (not leaked)
print('\nCorrelations with WASO (should be weak to moderate, NOT near 1.0):')
for c in ['lf_hf','rmssd','isi','whoqol','phq9','gad7','steps']:
    r, p = stats.pearsonr(df[c].fillna(df[c].mean()), df['waso_min'])
    flag = '⚠️ HIGH' if abs(r) > 0.6 else '✅'
    print(f'  {c:10s}: r={r:+.3f}  {flag}')

## ✂️ Cell 4 — Subject-Wise Train/Test Split (THE KEY FIX)
> Random split = same person in train and test → model memorises individual patterns → inflated accuracy.
> Subject-wise split = held-out participants never seen during training → honest generalisation.

In [ ]:
# 66 train participants, 16 test participants (≈80/20 by subject count)
all_pids   = df['pid'].unique()
rng2       = np.random.default_rng(SEED)
rng2.shuffle(all_pids)
n_train_p  = int(0.80 * len(all_pids))   # 66
train_pids = all_pids[:n_train_p]
test_pids  = all_pids[n_train_p:]

df_train = df[df['pid'].isin(train_pids)].reset_index(drop=True)
df_test  = df[df['pid'].isin(test_pids)].reset_index(drop=True)

print(f'Train: {len(train_pids)} participants, {len(df_train)} days')
print(f'Test : {len(test_pids)} participants, {len(df_test)} days')
print(f'✅ NO participant overlap between train and test')

# Sensor features to impute + scale
SENSOR_COLS = ['lf_hf','rmssd','steps','accel','gyro','lum']
ALL_FEAT    = ['lf_hf','rmssd','steps','accel','gyro','lum',
               'isi','whoqol','phq9','gad7','knhanes']

# KNN imputer fit ONLY on train
imputer = KNNImputer(n_neighbors=3)
df_train[SENSOR_COLS] = imputer.fit_transform(df_train[SENSOR_COLS])
df_test[SENSOR_COLS]  = imputer.transform(df_test[SENSOR_COLS])

# Scaler fit ONLY on train
scaler = StandardScaler()
df_train[ALL_FEAT] = scaler.fit_transform(df_train[ALL_FEAT])
df_test[ALL_FEAT]  = scaler.transform(df_test[ALL_FEAT])

print('✅ Imputation and scaling done (fit on train only, applied to test).')

## 🔬 Cell 5 — Leakage Audit: Correlation Sanity Check

In [ ]:
# Check correlation BEFORE scaling (raw values are more interpretable)
df_raw = pd.DataFrame({
    'lf_hf':lf_hf,'rmssd':rmssd,'steps':steps,'accel':accel,'gyro':gyro,
    'isi':isi_arr,'whoqol':whoqol_arr,'phq9':phq9_arr,'gad7':gad7_arr,
    'knhanes':knhanes_arr,'waso_min':waso,'waso':waso_binary
})

corr_with_target = df_raw[ALL_FEAT + ['waso_min']].corr()['waso_min'].drop('waso_min')
corr_with_target = corr_with_target.sort_values(key=abs, ascending=False)

print('Correlation with WASO (raw, before scaling):')
print('If any feature has |r| > 0.7 → suspect leakage')
for feat, r in corr_with_target.items():
    bar = '█' * int(abs(r)*30)
    flag = ' ⚠️ CHECK' if abs(r) > 0.6 else ''
    print(f'  {feat:10s}: {r:+.3f} {bar}{flag}')

print('\n✅ None should exceed ±0.6 in a clean dataset.')
print('LF/HF should be top correlator (paper: r=0.22).')

# Full correlation matrix
fig, ax = plt.subplots(figsize=(9,7))
mask = np.triu(np.ones_like(df_raw[ALL_FEAT].corr(), dtype=bool))
sns.heatmap(df_raw[ALL_FEAT].corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.4, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Matrix (pre-split)')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧪 Cell 6 — Leakage Test 1: Shuffle Labels
> If model still gets accuracy > 55% with shuffled labels, there is leakage.
> Expected honest result: ≈50% accuracy / 0.50 AUROC with shuffled labels.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

FEAT_B2 = ['lf_hf','rmssd','steps','accel','gyro','isi','whoqol']

X_tr_tab = df_train[FEAT_B2].values
y_tr     = df_train['waso'].values
X_te_tab = df_test[FEAT_B2].values
y_te     = df_test['waso'].values

# Real RF
rf_check = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED)
rf_check.fit(X_tr_tab, y_tr)
real_acc  = accuracy_score(y_te, rf_check.predict(X_te_tab))
real_auc  = roc_auc_score(y_te, rf_check.predict_proba(X_te_tab)[:,1])

# Shuffled labels RF
y_shuffled = y_tr.copy()
np.random.shuffle(y_shuffled)
rf_shuf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED)
rf_shuf.fit(X_tr_tab, y_shuffled)
shuf_acc = accuracy_score(y_te, rf_shuf.predict(X_te_tab))
shuf_auc = roc_auc_score(y_te, rf_shuf.predict_proba(X_te_tab)[:,1])

print('=== Shuffle-Label Leakage Test ===')
print(f'Real labels  → Acc={real_acc:.3f}  AUROC={real_auc:.3f}')
print(f'Shuffled     → Acc={shuf_acc:.3f}  AUROC={shuf_auc:.3f}  (should be ≈0.50)')
if shuf_auc < 0.58:
    print('✅ PASSED — shuffled result is near chance. No leakage from feature set.')
else:
    print('⚠️  Shuffled AUC is high — possible data leakage, investigate further.')

## 🧪 Cell 7 — Leakage Test 2: Without Questionnaire Features
> Sensor-only model. If performance doesn't drop much, questionnaires are NOT leaked.

In [ ]:
FEAT_SENSOR_ONLY = ['lf_hf','rmssd','steps','accel','gyro']

rf_sensor = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED)
rf_sensor.fit(df_train[FEAT_SENSOR_ONLY].values, y_tr)
s_acc = accuracy_score(y_te, rf_sensor.predict(df_test[FEAT_SENSOR_ONLY].values))
s_auc = roc_auc_score(y_te, rf_sensor.predict_proba(df_test[FEAT_SENSOR_ONLY].values)[:,1])

rf_full = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED)
rf_full.fit(df_train[FEAT_B2].values, y_tr)
f_acc = accuracy_score(y_te, rf_full.predict(df_test[FEAT_B2].values))
f_auc = roc_auc_score(y_te, rf_full.predict_proba(df_test[FEAT_B2].values)[:,1])

diff_auc = f_auc - s_auc

print('=== Feature Set Ablation Test ===')
print(f'Sensor-only (lf_hf,rmssd,steps,accel,gyro): Acc={s_acc:.3f}  AUROC={s_auc:.3f}')
print(f'B2 full (+ isi, whoqol):                     Acc={f_acc:.3f}  AUROC={f_auc:.3f}')
print(f'AUROC gain from adding questionnaires:        +{diff_auc:.3f}')
if diff_auc < 0.15:
    print('✅ Questionnaires add moderate signal — not leaking (gain < 0.15).')
else:
    print('⚠️  Large gain — questionnaire features may carry leakage.')

## 🪟 Cell 8 — Build 7-Day Sliding Window Sequences

In [ ]:
WINDOW    = 7
FEAT_B2   = ['lf_hf','rmssd','steps','accel','gyro','isi','whoqol']
N_FEAT    = len(FEAT_B2)

def make_sequences(df_in, feat_cols, window=7):
    X, y, pids = [], [], []
    for pid in df_in['pid'].unique():
        sub = df_in[df_in['pid']==pid].reset_index(drop=True)
        for t in range(window-1, len(sub)-1):
            win = sub.loc[t-window+1:t, feat_cols].values
            lbl = sub.loc[t+1, 'waso']
            if win.shape[0] == window:
                X.append(win); y.append(lbl); pids.append(pid)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), np.array(pids)

X_tr, y_tr_seq, _ = make_sequences(df_train, FEAT_B2, WINDOW)
X_te, y_te_seq, _ = make_sequences(df_test,  FEAT_B2, WINDOW)

# Flat versions for sklearn + ARIMA
X_tr_flat = X_tr.reshape(len(X_tr), -1)
X_te_flat = X_te.reshape(len(X_te), -1)
X_tr_last = X_tr[:, -1, :]    # last day only — for LIME/SHAP
X_te_last = X_te[:, -1, :]

print(f'Train sequences: X={X_tr.shape}  y={y_tr_seq.shape}')
print(f'Test  sequences: X={X_te.shape}  y={y_te_seq.shape}')
print(f'Class balance (train): 0={np.mean(y_tr_seq==0)*100:.1f}%  1={np.mean(y_tr_seq==1)*100:.1f}%')
print(f'Class balance (test) : 0={np.mean(y_te_seq==0)*100:.1f}%  1={np.mean(y_te_seq==1)*100:.1f}%')

## 📊 Cell 9 — EDA: LF/HF vs WASO (Figure 4 replica)

In [ ]:
# Use raw (unscaled) values for EDA
df_raw_eda = pd.DataFrame({'lf_hf':lf_hf,'rmssd':rmssd,'waso_min':waso,
                            'isi':isi_arr,'whoqol':whoqol_arr,'dow':day_of_week})

med        = df_raw_eda['lf_hf'].median()
lower_grp  = df_raw_eda[df_raw_eda['lf_hf'] <= med]['waso_min']
higher_grp = df_raw_eda[df_raw_eda['lf_hf'] >  med]['waso_min']

ks_w = kstest(df_raw_eda['waso_min'], 'norm',
              args=(df_raw_eda['waso_min'].mean(), df_raw_eda['waso_min'].std()))
stat, pval = stats.ranksums(lower_grp, higher_grp)
r_corr, p_corr = stats.pearsonr(df_raw_eda['lf_hf'], df_raw_eda['waso_min'])

print('=== Statistical Tests ===')
print(f'KS normality test WASO: p={ks_w.pvalue:.4f} → {"Non-normal" if ks_w.pvalue<0.05 else "Normal"}')
print(f'Wilcoxon Rank-Sum: W={stat:.2f}, p={pval:.4f} (paper: p=0.012)')
print(f'Lower  LF/HF: WASO={lower_grp.mean():.1f}±{lower_grp.std():.1f} min  (paper: 7.5±2.0)')
print(f'Higher LF/HF: WASO={higher_grp.mean():.1f}±{higher_grp.std():.1f} min  (paper: 14.9±3.0)')
print(f'Pearson r(LF/HF, WASO)={r_corr:.3f}, p={p_corr:.4f}  (paper: r=0.22)')

fig, axes = plt.subplots(1, 3, figsize=(15,5))

# Fig 4 replica
axes[0].barh(['Lower LF/HF','Higher LF/HF'],
             [lower_grp.mean(), higher_grp.mean()],
             xerr=[lower_grp.std(), higher_grp.std()],
             color=['#1D9E75','#E24B4A'], capsize=5, alpha=0.85, height=0.45)
axes[0].axvline(df_raw_eda['waso_min'].mean(), color='gray', linestyle='--',
                label=f'Mean: {df_raw_eda.waso_min.mean():.1f}')
axes[0].set_xlabel('Mean WASO (min)'); axes[0].set_title('LF/HF Group vs WASO (Fig.4 replica)')
axes[0].legend()
if pval < 0.05:
    axes[0].text(higher_grp.mean()+higher_grp.std()+0.3, 1.0, '* p<0.05', va='center')

# Scatter
axes[1].scatter(df_raw_eda['lf_hf'], df_raw_eda['waso_min'], alpha=0.12, s=7, color='#378ADD')
z = np.polyfit(df_raw_eda['lf_hf'], df_raw_eda['waso_min'], 1)
xl = np.linspace(df_raw_eda['lf_hf'].min(), df_raw_eda['lf_hf'].max(), 100)
axes[1].plot(xl, np.poly1d(z)(xl), 'r-', lw=2)
axes[1].set_xlabel('LF/HF Ratio'); axes[1].set_ylabel('WASO (min)')
axes[1].set_title(f'LF/HF vs WASO  r={r_corr:.2f}')

# Day of week
dow_means = [df_raw_eda[df_raw_eda['dow']==d]['waso_min'].mean() for d in range(7)]
axes[2].bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], dow_means,
            color=['#378ADD']*5+['#E24B4A','#E24B4A'], alpha=0.85)
axes[2].set_ylabel('Mean WASO (min)'); axes[2].set_title('WASO by Day (weekends higher)')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA done.')

## 📏 Cell 10 — Results Helper + Storage

In [ ]:
results = []
roc_store = {}

def evaluate(name, y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    loss = log_loss(y_true, np.clip(y_prob, 1e-7, 1-1e-7))
    results.append({'Model':name,'Accuracy':round(acc,4),'Precision':round(prec,4),
                    'Recall':round(rec,4),'AUROC':round(auc,4),'Loss':round(loss,4)})
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_store[name] = (fpr, tpr, auc)
    print(f'  [{name:20s}] Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc:.4f} Loss={loss:.4f}')
    return acc, prec, rec, auc, loss

print('✅ Evaluation helper ready.')

## 📐 Cell 11 — Model 1: ARIMA (Statistical baseline)

In [ ]:
print('Training ARIMA...')
arima_probs, arima_true = [], []

for pid in test_pids:
    sub = df[df['pid']==pid].reset_index(drop=True)
    ts  = sub['lf_hf'].fillna(sub['lf_hf'].mean()).values
    yt  = sub['waso'].values
    if len(ts) < 12:
        continue
    try:
        # Train on test participant's own history (no train-set data → no leakage)
        split = max(8, int(len(ts)*0.75))
        model = ARIMA(ts[:split], order=(2,0,1)).fit()
        preds = model.forecast(len(ts)-split)
        mn, mx = preds.min(), preds.max()
        probs = (preds-mn)/(mx-mn+1e-8)
        probs = np.clip(probs, 0.02, 0.98)
        arima_probs.extend(probs)
        arima_true.extend(yt[split:])
    except Exception:
        continue

arima_probs = np.array(arima_probs)
arima_true  = np.array(arima_true)

fpr_a, tpr_a, thr_a = roc_curve(arima_true, arima_probs)
opt_thr = thr_a[np.argmax(tpr_a - fpr_a)]
arima_preds = (arima_probs >= opt_thr).astype(int)
roc_store['ARIMA'] = (fpr_a, tpr_a, roc_auc_score(arima_true, arima_probs))

acc  = accuracy_score(arima_true, arima_preds)
prec = precision_score(arima_true, arima_preds, zero_division=0)
rec  = recall_score(arima_true, arima_preds, zero_division=0)
auc  = roc_auc_score(arima_true, arima_probs)
loss = log_loss(arima_true, np.clip(arima_probs,1e-7,1-1e-7))
results.append({'Model':'ARIMA','Accuracy':round(acc,4),'Precision':round(prec,4),
                'Recall':round(rec,4),'AUROC':round(auc,4),'Loss':round(loss,4)})
print(f'  [ARIMA               ] Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc:.4f} Loss={loss:.4f}')
print('✅ ARIMA done.')

## 🌲 Cell 12 — Model 2: Random Forest

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf.fit(X_tr_last, y_tr_seq)
rf_prob = rf.predict_proba(X_te_last)[:,1]
rf_pred = rf.predict(X_te_last)
print('\nRandom Forest:')
evaluate('Random Forest', y_te_seq, rf_pred, rf_prob)
print('✅ RF done.')

## ⚡ Cell 13 — Model 3: XGBoost

In [ ]:
print('Training XGBoost...')
neg, pos = np.bincount(y_tr_seq)
scale_pos = neg / pos
xgb = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric='logloss', random_state=SEED, tree_method='hist',
    verbosity=0
)
xgb.fit(X_tr_flat, y_tr_seq,
        eval_set=[(X_te_flat, y_te_seq)], verbose=False)
xgb_prob = xgb.predict_proba(X_te_flat)[:,1]
xgb_pred = xgb.predict(X_te_flat)
print('\nXGBoost:')
evaluate('XGBoost', y_te_seq, xgb_pred, xgb_prob)
print('✅ XGBoost done.')

## 🔄 Cell 14 — Model 4: GRU  [Neural Network #1]

In [ ]:
# ── GRU is a Neural Network (Recurrent) ──────────────────────────────────────
# Architecture: Input → GRU(64) → GRU(32) → Dense(32,relu) → Dropout → Dense(1,sigmoid)
print('Training GRU (Recurrent Neural Network)...')

def build_gru(win, nf):
    inp = keras.Input(shape=(win, nf))
    x = layers.GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)(inp)
    x = layers.GRU(32, dropout=0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

cb = [EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
      ReduceLROnPlateau(patience=4, factor=0.5, min_lr=1e-6)]

gru = build_gru(WINDOW, N_FEAT)
gru.fit(X_tr, y_tr_seq, validation_split=0.15,
        epochs=80, batch_size=32, callbacks=cb, verbose=0)
gru_prob = gru.predict(X_te, verbose=0).flatten()
gru_pred = (gru_prob >= 0.5).astype(int)
print('\nGRU (Neural Network):')
evaluate('GRU', y_te_seq, gru_pred, gru_prob)
print('✅ GRU done.')

## 🌊 Cell 15 — Model 5: TCN  [Neural Network #2]

In [ ]:
# ── TCN is a Neural Network (Convolutional + Temporal) ───────────────────────
# Architecture: Input → Dilated Conv1D (rates 1,2,4) → BN → GAP → Dense → Dropout → Output
print('Training TCN (Temporal Convolutional Neural Network)...')

def build_tcn(win, nf):
    inp = keras.Input(shape=(win, nf))
    x = layers.Conv1D(64, 2, dilation_rate=1, padding='causal', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 2, dilation_rate=2, padding='causal', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(32, 2, dilation_rate=4, padding='causal', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

tcn = build_tcn(WINDOW, N_FEAT)
tcn.fit(X_tr, y_tr_seq, validation_split=0.15,
        epochs=80, batch_size=32, callbacks=cb, verbose=0)
tcn_prob = tcn.predict(X_te, verbose=0).flatten()
tcn_pred = (tcn_prob >= 0.5).astype(int)
print('\nTCN (Neural Network):')
evaluate('TCN', y_te_seq, tcn_pred, tcn_prob)
print('✅ TCN done.')

## 🔭 Cell 16 — Model 6: Transformer  [Neural Network #3]

In [ ]:
# ── Transformer is a Neural Network (Attention-based) ────────────────────────
# Architecture: Input → Linear projection → 2× Encoder block (MHA + FFN + LN) → GAP → Dense → Output
print('Training Transformer (Attention-based Neural Network)...')

def enc_block(x, heads, head_dim, ff_dim, dr):
    attn = layers.MultiHeadAttention(num_heads=heads, key_dim=head_dim, dropout=dr)(x, x)
    x    = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dr)(attn))
    ff   = layers.Dense(ff_dim, activation='gelu')(x)
    ff   = layers.Dense(x.shape[-1])(layers.Dropout(dr)(ff))
    return layers.LayerNormalization(epsilon=1e-6)(x + ff)

def build_transformer(win, nf):
    inp = keras.Input(shape=(win, nf))
    x   = layers.Dense(64)(inp)
    x   = enc_block(x, heads=4, head_dim=16, ff_dim=64, dr=0.1)
    x   = enc_block(x, heads=4, head_dim=16, ff_dim=64, dr=0.1)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(32, activation='relu')(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m   = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(5e-4),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

transformer = build_transformer(WINDOW, N_FEAT)
transformer.fit(X_tr, y_tr_seq, validation_split=0.15,
                epochs=80, batch_size=32, callbacks=cb, verbose=0)
tr_prob = transformer.predict(X_te, verbose=0).flatten()
tr_pred = (tr_prob >= 0.5).astype(int)
print('\nTransformer (Neural Network):')
evaluate('Transformer', y_te_seq, tr_pred, tr_prob)
print('✅ Transformer done.')

## ⭐ Cell 17 — Model 7: LSTM  [Neural Network #4 — Best Model]

In [ ]:
# ── LSTM is a Neural Network (Recurrent with memory gates) ───────────────────
# Architecture: Input → LSTM(128) → LSTM(64) → LSTM(32) → BN → Dense(64) → Dropout → Dense(1)
# LSTM gates: input gate, forget gate, output gate — learn what to remember across 7 days
print('Training LSTM (Long Short-Term Memory Neural Network) — best model from paper...')

def build_lstm(win, nf):
    inp = keras.Input(shape=(win, nf))
    x   = layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)(inp)
    x   = layers.LSTM(64,  return_sequences=True, dropout=0.2, recurrent_dropout=0.1)(x)
    x   = layers.LSTM(32,  dropout=0.2)(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1,  activation='sigmoid')(x)
    m   = Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

cb_lstm = [
    EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(patience=6, factor=0.5, min_lr=1e-6)
]

lstm = build_lstm(WINDOW, N_FEAT)
lstm.summary()

history = lstm.fit(
    X_tr, y_tr_seq,
    validation_split=0.15,
    epochs=120, batch_size=32,
    callbacks=cb_lstm, verbose=1
)

lstm_prob = lstm.predict(X_te, verbose=0).flatten()
lstm_pred = (lstm_prob >= 0.5).astype(int)
print('\n⭐ LSTM (Neural Network — best model):')
evaluate('LSTM', y_te_seq, lstm_pred, lstm_prob)
print('\nPaper reported: Acc=0.904 | Prec=0.913 | Rec=0.899 | AUC=0.901')
print('Our results use subject-wise split — honest generalisation, may be slightly lower.')

## 📉 Cell 18 — LSTM Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
ep = range(1, len(history.history['loss'])+1)

axes[0].plot(ep, history.history['loss'],     label='Train', color='#378ADD')
axes[0].plot(ep, history.history['val_loss'], label='Val',   color='#E24B4A', ls='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('LSTM — Loss'); axes[0].legend()

axes[1].plot(ep, [v*100 for v in history.history['accuracy']],     label='Train', color='#1D9E75')
axes[1].plot(ep, [v*100 for v in history.history['val_accuracy']], label='Val',   color='#EF9F27', ls='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('LSTM — Accuracy'); axes[1].legend()

plt.tight_layout()
plt.savefig('lstm_training.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Cell 19 — Full Results Table + Charts

In [ ]:
res_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print('='*72)
print('FINAL RESULTS TABLE  (subject-wise split — no leakage)')
print('='*72)
print(res_df.to_string(index=False))
print('='*72)
print('Paper (random split, may be optimistic): LSTM Acc=0.904 AUC=0.901')
print('Our results (subject-wise) should be lower but HONEST.')
print('Typical honest LSTM performance on this type of data: Acc 0.72-0.85, AUC 0.72-0.85')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
metrics = ['Accuracy','Precision','Recall','AUROC','Loss']

for i, m in enumerate(metrics):
    sd = res_df.sort_values(m, ascending=(m=='Loss'))
    colors = ['#1D9E75' if n=='LSTM' else
              '#7F77DD' if n in ('GRU','TCN','Transformer') else '#378ADD'
              for n in sd['Model']]
    axes[i].barh(sd['Model'], sd[m], color=colors, alpha=0.85, edgecolor='white')
    axes[i].set_title(m)
    for j,(v,n) in enumerate(zip(sd[m], sd['Model'])):
        axes[i].text(v+0.003 if m!='Loss' else v+0.005, j, f'{v:.3f}', va='center', fontsize=9)

# Legend for colors
from matplotlib.patches import Patch
legend = [Patch(color='#1D9E75', label='LSTM (best NN)'),
          Patch(color='#7F77DD', label='Other NNs (GRU/TCN/Transformer)'),
          Patch(color='#378ADD', label='Non-NN (ARIMA/RF/XGB)')]
axes[4].legend(handles=legend, loc='lower right', fontsize=9)

# ROC curves
ax_roc = axes[5]
cols_roc = ['#E24B4A','#1D9E75','#378ADD','#EF9F27','#7F77DD','#D85A30','#888780']
for (name,(fpr,tpr,auc_v)), col in zip(roc_store.items(), cols_roc):
    ax_roc.plot(fpr, tpr, label=f'{name} ({auc_v:.3f})', color=col, lw=1.5)
ax_roc.plot([0,1],[0,1],'k--', alpha=0.4)
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title('ROC Curves')
ax_roc.legend(fontsize=8)

plt.suptitle('Model Comparison — Subject-Wise Split (Honest Results)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Charts saved.')

## 🔍 Cell 20 — Confusion Matrix + Report (LSTM)

In [ ]:
print('LSTM Classification Report:')
print(classification_report(y_te_seq, lstm_pred,
                             target_names=['No awakening','Awakening']))
cm = confusion_matrix(y_te_seq, lstm_pred)
fig, ax = plt.subplots(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'])
ax.set_title('LSTM Confusion Matrix')
plt.tight_layout()
plt.savefig('lstm_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 💡 Cell 21 — LIME Analysis (Paper replication, Figure 5)

In [ ]:
print('Running LIME (as in paper, Figure 5)...')

feat_names_flat = [f'{f}_d{d+1}' for d in range(WINDOW) for f in FEAT_B2]

def lstm_pred_fn(X_flat):
    X3 = X_flat.reshape(-1, WINDOW, N_FEAT)
    p  = lstm.predict(X3, verbose=0).flatten()
    return np.column_stack([1-p, p])

lime_exp = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_tr_flat,
    feature_names=feat_names_flat,
    class_names=['No awakening','Awakening'],
    mode='classification', random_state=SEED
)

lime_imp = np.zeros(len(feat_names_flat))
N_LIME = 40
sample_idx = np.concatenate([
    np.where(y_te_seq==0)[0][:20],
    np.where(y_te_seq==1)[0][:20]
])

for idx in sample_idx:
    ex = lime_exp.explain_instance(X_te_flat[idx], lstm_pred_fn,
                                   num_features=len(feat_names_flat), num_samples=150)
    for feat_str, w in ex.as_list(label=1):
        for k, fn in enumerate(feat_names_flat):
            if fn in feat_str or feat_str in fn:
                lime_imp[k] += abs(w); break

lime_imp /= N_LIME

# Aggregate by feature name (sum over 7 days)
lime_by_feat = {f: 0.0 for f in FEAT_B2}
for k, fn in enumerate(feat_names_flat):
    for f in FEAT_B2:
        if fn.startswith(f):
            lime_by_feat[f] += lime_imp[k]; break

lime_ser = pd.Series(lime_by_feat).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8,5))
colors = ['#1D9E75' if v==lime_ser.max() else '#378ADD' for v in lime_ser.values]
ax.barh(lime_ser.index, lime_ser.values, color=colors, alpha=0.85, edgecolor='white')
ax.set_xlabel('Mean |LIME weight|')
ax.set_title('LIME Feature Importance — LSTM (replicates Fig.5)')
plt.tight_layout()
plt.savefig('lime_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLIME ranking (most → least):')
for f,v in lime_ser.sort_values(ascending=False).items():
    print(f'  {f:12s}: {v:.5f}')
print('\nPaper top 3: lf_hf → ISI → whoqol')
print('✅ LIME done.')

## ✨ Cell 22 — SHAP Analysis (Our Extra Contribution)

In [ ]:
print('Running SHAP (our contribution beyond the paper)...')

# SHAP TreeExplainer on RF (exact + fast)
rf_shap = RandomForestClassifier(n_estimators=200, max_depth=8,
                                  random_state=SEED, n_jobs=-1)
rf_shap.fit(X_tr_last, y_tr_seq)
shap_exp   = shap.TreeExplainer(rf_shap)
shap_vals  = shap_exp.shap_values(X_te_last)   # list of 2 arrays
shap_cls1  = shap_vals[1]                       # class 1 (awakening)

shap_mean  = np.abs(shap_cls1).mean(axis=0)
shap_ser   = pd.Series(shap_mean, index=FEAT_B2).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar
colors_s = ['#1D9E75' if v==shap_ser.max() else '#7F77DD' for v in shap_ser.values]
axes[0].barh(shap_ser.index, shap_ser.values, color=colors_s, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Mean |SHAP value|')
axes[0].set_title('SHAP Global Feature Importance\n(our contribution)')

# Beeswarm
plt.sca(axes[1])
shap.summary_plot(shap_cls1, X_te_last, feature_names=FEAT_B2, show=False, plot_size=None)
axes[1].set_title('SHAP Beeswarm — Class 1 (Awakening)')

plt.tight_layout()
plt.savefig('shap_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare LIME vs SHAP rankings
lime_rank = lime_ser.sort_values(ascending=False).index.tolist()
shap_rank = shap_ser.sort_values(ascending=False).index.tolist()
comp = pd.DataFrame({'LIME rank': lime_rank, 'SHAP rank': shap_rank})
comp.index = [f'#{i+1}' for i in range(len(comp))]
print('\nLIME vs SHAP Feature Ranking Comparison:')
print(comp.to_string())
print('\n✅ SHAP done. LIME+SHAP comparison is our key contribution beyond the paper!')

## 🧪 Cell 23 — Permutation Importance (ChatGPT recommended check)

In [ ]:
from sklearn.inspection import permutation_importance

print('Permutation importance test (check which features the model truly relies on)...')
pi = permutation_importance(rf_shap, X_te_last, y_te_seq,
                             n_repeats=15, random_state=SEED, n_jobs=-1)

pi_df = pd.DataFrame({'feature': FEAT_B2,
                       'importance_mean': pi.importances_mean,
                       'importance_std':  pi.importances_std
                      }).sort_values('importance_mean', ascending=False)

print('\nPermutation Importance (mean accuracy drop when feature is shuffled):')
print(pi_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,5))
colors_pi = ['#1D9E75' if f==pi_df.iloc[0]['feature'] else '#378ADD'
             for f in pi_df['feature']]
ax.barh(pi_df['feature'], pi_df['importance_mean'],
        xerr=pi_df['importance_std'], color=colors_pi, alpha=0.85,
        capsize=4, edgecolor='white')
ax.set_xlabel('Mean accuracy drop when feature is shuffled')
ax.set_title('Permutation Importance — If lf_hf is top → model learned real signal')
plt.tight_layout()
plt.savefig('permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ If lf_hf / rmssd have highest permutation importance → model learned genuine signal.')
print('   If isi/whoqol dominate → possible questionnaire over-reliance.')

## 📋 Cell 24 — Final Summary + Save Files

In [ ]:
import os, zipfile

print('='*70)
print('COMPLETE SUMMARY')
print('='*70)

print('\n📌 Neural Networks used in this project:')
print('  ✅ GRU        — Gated Recurrent Unit (recurrent NN)')
print('  ✅ TCN        — Temporal Convolutional Network (convolutional NN)')
print('  ✅ Transformer — Multi-head self-attention NN')
print('  ✅ LSTM       — Long Short-Term Memory (recurrent NN) — BEST')
print('  ❌ ARIMA      — Statistical, not a neural network')
print('  ❌ Random Forest — Ensemble trees, not a neural network')
print('  ❌ XGBoost    — Gradient boosted trees, not a neural network')

print('\n🔒 Leakage Fixes Applied:')
print('  ✅ Subject-wise train/test split (no participant overlap)')
print('  ✅ Scaler/imputer fit only on training set')
print('  ✅ Questionnaire features sampled independently (not from WASO)')
print('  ✅ Shuffle-label test included')
print('  ✅ Feature ablation test included')
print('  ✅ Permutation importance test included')

print('\n📊 Final Results (honest, subject-wise):')
print(res_df.to_string(index=False))

best = res_df.iloc[0]
print(f'\n🏆 Best model: {best["Model"]}')
print(f'   Accuracy={best["Accuracy"]}  Precision={best["Precision"]}  Recall={best["Recall"]}  AUROC={best["AUROC"]}')

print('\n✨ Our contributions beyond the paper:')
print('  1. SHAP alongside LIME — global + local explainability comparison')
print('  2. Permutation importance — verifies model relies on real signal')
print('  3. Subject-wise split — more honest than paper\'s random split')
print('  4. Leakage audit cells (shuffle test + ablation test)')

# Save CSV
res_df.to_csv('model_results.csv', index=False)

# Save LSTM
lstm.save('lstm_model.keras')

print('\n📁 Saved files:')
files_out = ['eda_plots.png','correlation_matrix.png','lstm_training.png',
             'model_comparison.png','lstm_confusion.png',
             'lime_plot.png','shap_plot.png','permutation_importance.png',
             'model_results.csv','lstm_model.keras']
for f in files_out:
    print(f'  {"✅" if os.path.exists(f) else "❌"} {f}')

## 📥 Cell 25 — Download ZIP (run last)

In [ ]:
from google.colab import files
import zipfile, os

zpath = 'sleep_project_results.zip'
all_files = ['eda_plots.png','correlation_matrix.png','lstm_training.png',
             'model_comparison.png','lstm_confusion.png',
             'lime_plot.png','shap_plot.png','permutation_importance.png',
             'model_results.csv']

with zipfile.ZipFile(zpath, 'w') as zf:
    for f in all_files:
        if os.path.exists(f):
            zf.write(f)

files.download(zpath)
print(f'✅ Downloaded: {zpath}')